# 01 — Data Generation

## Purpose
Generate a synthetic resume dataset with realistic correlations between latent age group and observable resume features, then persist the dataset and generation metadata to disk for reproducible experiments.

## Outputs
- `data/synthetic_resumes_full.parquet`
- `data/synthetic_resumes_sample.csv`
- `data/generation_metadata.json`

## Notes
- Age (or age_group) is generated as a hidden/protected attribute for analysis only.
- Training features will exclude age fields by design.
- Bias in the simulated callback label is adjustable via a configuration parameter.

## Imports and Configuration

In [1]:
import sys
from pathlib import Path

# Add project root to Python path
PROJECT_ROOT = Path().resolve().parents[0]
sys.path.append(str(PROJECT_ROOT))

print("Project root added to path:", PROJECT_ROOT)

Project root added to path: /home/marshall/Projects/ucbaicert/ucbai-cs-resumefilter


In [2]:
import importlib
import src.data_generation as dg
importlib.reload(dg)

<module 'src.data_generation' from '/home/marshall/Projects/ucbaicert/ucbai-cs-resumefilter/src/data_generation.py'>

## Resume Feature Schema and Controlled Vocabularies

In [3]:
from src.data_generation import RESUME_SCHEMA_DTYPES, DEFAULT_CATEGORY_LEVELS

print("Columns:", list(RESUME_SCHEMA_DTYPES.keys()))
print("Role levels:", DEFAULT_CATEGORY_LEVELS["target_role_level"])

Columns: ['candidate_id', 'application_year', 'target_role_family', 'target_role_level', 'region', 'highest_degree', 'graduation_year', 'school_tier', 'gpa_bucket', 'years_experience_total', 'years_experience_relevant', 'num_employers', 'avg_tenure_years', 'months_since_last_role', 'num_gaps_over_6mo', 'most_recent_title', 'most_recent_company_size', 'management_years', 'reports_max', 'num_skills_listed', 'num_programming_languages', 'num_cloud_platforms', 'num_databases', 'skill_python', 'skill_java', 'skill_javascript', 'skill_go', 'skill_kubernetes', 'skill_aws', 'skill_gcp', 'skill_azure', 'skill_sql', 'skill_spark', 'skill_terraform', 'skill_linux', 'skill_ml', 'legacy_tech_count', 'modern_tech_count', 'cert_count', 'has_top_cloud_cert', 'github_url_present', 'portfolio_url_present', 'open_source_mentions', 'patent_count', 'resume_word_count', 'bullet_count', 'quantified_impact_count', 'keyword_match_score', 'format_clean_score', 'salary_expectation_usd', 'willing_to_relocate', 'r

## Synthetic Data Generator

In [4]:
from src.data_generation import generate_synthetic_resumes
help(generate_synthetic_resumes)

Help on function generate_synthetic_resumes in module src.data_generation:

generate_synthetic_resumes(config: 'GenerationConfig') -> 'pd.DataFrame'
    Generate a synthetic resume dataset designed to study proxy-bias leakage.

    - age/age_group is generated for fairness evaluation only (not for model features).
    - graduation_year is included as a configurable strong proxy.
    - callback label includes an adjustable penalty for age >= 40 groups.

    Returns:
        DataFrame with columns matching ALL_DTYPES (where applicable).



## Generate Dataset

The synthetic label-generation process was tuned to produce a measurable but non-degenerate age-patterned disparity across groups. The resulting callback distribution provides sufficient variation for downstream fairness evaluation while avoiding complete collapse of older candidate groups.

In [5]:
config = dg.GenerationConfig(
    n_samples=10_000,
    bias_strength=1.0,
    seed=42,
    application_year=2025,
    callback_quantile=0.50
)

df = dg.generate_synthetic_resumes(config)

df.groupby("age_group", observed=False)["callback"].mean().sort_index()

age_group
<30       0.57463
30-39    0.703941
40-49    0.457212
50-59    0.184705
60+         0.102
Name: callback, dtype: Float64

## Basic Validation and Sanity Checks

In [6]:
df.shape
df["age_group"].value_counts(normalize=True)
df.groupby("age_group", observed=False)["callback"].mean()
df.describe()

,application_year,true_age,school_tier,graduation_year,years_experience_total,years_experience_relevant,num_employers,avg_tenure_years,months_since_last_role,num_gaps_over_6mo,...,resume_word_count,bullet_count,quantified_impact_count,keyword_match_score,format_clean_score,salary_expectation_usd,estimated_start_year,tech_recency_score,leadership_signal_score,stability_score
count,10000.0,10000.000000,10000.00000,10000.00000,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.00000,...,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000
mean,2025.0,39.268100,3.02250,2007.73910,10.325300,8.776296,2.68740,2.640116,5.261300,0.53110,...,650.230700,27.716000,3.983900,0.717265,0.800722,154570.876500,2014.67880,0.571359,0.094512,0.276148
std,0.0,11.501602,1.41619,11.53914,8.498088,7.308032,2.58572,1.013098,5.873097,0.71266,...,180.778185,11.746844,1.991943,0.153091,0.114817,68350.545958,8.50285,0.091896,0.101070,0.126063
min,2025.0,22.000000,1.00000,1980.00000,0.000000,0.000000,0.00000,0.500000,0.000000,0.00000,...,200.000000,5.000000,0.000000,0.450053,0.600012,50000.000000,1995.00000,0.281215,0.000000,-0.121667
25%,2025.0,30.000000,2.00000,1999.00000,2.729886,2.278005,0.00000,1.939528,0.000000,0.00000,...,527.000000,20.000000,3.000000,0.586632,0.702062,97852.250000,2009.00000,0.508102,0.006165,0.190201
50%,2025.0,38.000000,3.00000,2009.00000,9.096188,7.696482,2.00000,2.636937,4.000000,0.00000,...,650.000000,28.000000,4.000000,0.718014,0.802312,142137.000000,2016.00000,0.569000,0.061338,0.275706
75%,2025.0,48.000000,4.00000,2017.00000,16.469222,13.920844,4.00000,3.324044,9.000000,1.00000,...,773.250000,36.000000,5.000000,0.851798,0.900718,203343.750000,2022.00000,0.631600,0.153126,0.362287
max,2025.0,66.000000,5.00000,2026.00000,30.000000,29.943209,13.00000,6.401647,36.000000,5.00000,...,1366.000000,74.000000,13.000000,0.979994,0.999876,356541.000000,2025.00000,0.882431,0.502549,0.746859


## Save Artifacts (Dataset + Metadata)

In [7]:
save_artifacts(
    df,
    out_dir=PROJECT_ROOT / "data" / "baseline",
    config=config
)

NameError: name 'save_artifacts' is not defined